# 8.04 真机落地清单：从"跑通"到"可交付"

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/08_vla_rl/8.04_real_world.ipynb)


## 为什么要学

仿真和玩具任务里，一个实验几分钟就能跑完，评估 200 个回合不费吹灰之力。真机上，**每一次评估都很贵**：一个回合 30 秒到几分钟，还要人工复位。所以真机工作的核心能力不是"会调参"，而是：

1. **用尽量少的试验，得出可靠的结论**（统计）；
2. **系统地记录和比较实验**（工程习惯）；
3. **把大问题拆成有明确过关标准的阶段**（规划）。

本课给出一套可以直接用在线头对插任务上的方法，并配两个小工具。

---

## 1. 评测协议：先定规则，再做实验

在动手调任何东西之前，先写下**评测协议**，并且之后所有实验都严格遵守它：

| 项目 | 例子（线头对插） |
|---|---|
| 初始条件分布 | 线头位置在标记区域内（如 4cm × 4cm）随机；插座位置固定；光照：实验室常规 |
| 每次评测的回合数 | 固定 20 次（快速筛选）/ 50 次（重要结论） |
| 成功判定 | 插入到位并保持 2 秒；**由同一个人按同一标准判断** |
| 超时 | 60 秒 |
| 失败分类 | 没抓到 / 抓偏 / 对不准 / 插一半 / 插歪 / 碰撞 / 超时 |
| 记录 | 每个回合：成功与否、失败类型、耗时、是否人工干预、视频 |

**失败分类比成功率更重要**。它告诉你下一步该做什么：

- 大部分失败是"没抓到"：问题在前段，补示教数据或改相机视角，**RL 精修帮不上忙**；
- 大部分失败是"对不准、插一半"：正是 RLT 擅长的"最后一厘米"；
- 失败集中在某个初始区域：数据覆盖不足。

## 2. 统计：20 次里成功 16 次，到底说明了什么？

成功率是一个二项分布的估计，**样本少时误差非常大**。用 **Wilson 置信区间** 估计真实成功率的范围（比"正负标准差"在小样本、接近 0 或 1 时可靠得多）：

$$
\hat p=\frac{k}{n},\qquad
\text{CI}=\frac{\hat p+\frac{z^2}{2n}\pm z\sqrt{\frac{\hat p(1-\hat p)}{n}+\frac{z^2}{4n^2}}}{1+\frac{z^2}{n}},\qquad z=1.96\ (95\%)
$$

### 练习 1：Wilson 置信区间

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def wilson_ci(k, n, z=1.96):
    """k 次成功 / n 次试验 -> (下界, 上界)"""
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
import math
import numpy as np
import matplotlib.pyplot as plt

def wilson_ci(k, n, z=1.96):
    """k 次成功 / n 次试验 -> (下界, 上界)"""
    if n == 0:
        return 0.0, 1.0
    p = k / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return max(0.0, center - half), min(1.0, center + half)

In [ ]:
# ✅ 自测
lo, hi = wilson_ci(16, 20)
print(f"16/20：成功率 0.80，95% 置信区间 [{lo:.2f}, {hi:.2f}]")
assert abs(lo - 0.584) < 0.01 and abs(hi - 0.919) < 0.01
for k, n in [(18, 20), (45, 50), (90, 100), (20, 20)]:
    lo, hi = wilson_ci(k, n); print(f"{k}/{n}: [{lo:.2f}, {hi:.2f}]")
print("通过 ✅")

**读结果**：20 次中成功 16 次，真实成功率可能在 58%~92% 之间。所以"改了一个参数，从 16/20 变成 18/20"**很可能只是运气**。

### 练习 2：两个版本真的有差别吗？

用"两个比例的差"的置信区间（这里用简单的正态近似 + 连续性修正就够了），判断改进是否可信。也可以用一个更直观的办法：**自助法（bootstrap）**。

In [ ]:
def bootstrap_diff(k1, n1, k2, n2, n_boot=20000, seed=0):
    """返回 P(版本 2 的真实成功率 > 版本 1)。做法：分别从两个版本的 0/1 结果中有放回地重采样，比较成功率"""
    rng = np.random.default_rng(seed)
    # TODO: a = [1]*k1 + [0]*(n1-k1)；b 同理；重复 n_boot 次：rng.choice(a, n1).mean() 与 rng.choice(b, n2).mean() 比较
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def bootstrap_diff(k1, n1, k2, n2, n_boot=20000, seed=0):
    """返回 P(版本 2 的真实成功率 > 版本 1)。做法：分别从两个版本的 0/1 结果中有放回地重采样，比较成功率"""
    rng = np.random.default_rng(seed)
    a = np.array([1] * k1 + [0] * (n1 - k1)); b = np.array([1] * k2 + [0] * (n2 - k2))
    pa = rng.choice(a, size=(n_boot, n1)).mean(1)
    pb = rng.choice(b, size=(n_boot, n2)).mean(1)
    return float((pb > pa).mean())

In [ ]:
# ✅ 自测
for (k1, n1, k2, n2) in [(16, 20, 18, 20), (16, 20, 20, 20), (32, 50, 45, 50)]:
    print(f"{k1}/{n1} -> {k2}/{n2}：版本 2 更好的概率 ≈ {bootstrap_diff(k1, n1, k2, n2):.2f}")
assert bootstrap_diff(32, 50, 45, 50) > 0.95
print("通过 ✅ 经验法则：想区分 80% 和 90%，每个版本至少需要 50~100 次试验")

In [ ]:
# 多少次试验才够？画出置信区间宽度随 n 的变化
ns = np.arange(5, 201, 5)
widths = [np.subtract(*wilson_ci(int(0.8 * n), n)[::-1]) for n in ns]
plt.figure(figsize=(5, 3)); plt.plot(ns, widths); plt.xlabel("number of trials"); plt.ylabel("95% CI width (p=0.8)"); plt.grid(alpha=.3); plt.show()

## 3. 实验日志：最朴素也最有用的工具

每做一次评测，写一行。坚持三个月，你就有了一份"自己的论文"。

In [ ]:
import csv, os, datetime

LOG = "experiment_log.csv"
FIELDS = ["date", "name", "ckpt", "change", "n", "success", "ci_low", "ci_high", "top_failure", "notes"]

def log_experiment(name, ckpt, change, outcomes, failures, notes=""):
    """outcomes: 0/1 列表；failures: 失败类型列表（与失败的回合对应）"""
    k, n = int(sum(outcomes)), len(outcomes)
    lo, hi = wilson_ci(k, n)
    top = max(set(failures), key=failures.count) if failures else ""
    row = dict(date=datetime.date.today().isoformat(), name=name, ckpt=ckpt, change=change, n=n, success=round(k / n, 3),
               ci_low=round(lo, 3), ci_high=round(hi, 3), top_failure=top, notes=notes)
    new = not os.path.exists(LOG)
    with open(LOG, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    return row

print(log_experiment("sft_baseline", "pi05_wire/30000", "基线", [1] * 13 + [0] * 7, ["对不准"] * 4 + ["插一半"] * 2 + ["没抓到"]))
print(log_experiment("rlt_v1", "rlt/step2000", "RLT, bc=0.01, std=0.002rad", [1] * 17 + [0] * 3, ["对不准", "插一半", "超时"]))

---

## 4. 线头对插：一个分阶段的方案

每个阶段都有**过关标准**；达不到就不进入下一阶段。

### 阶段 0：评测协议与基础设施（1 周）

- [ ] 写下第 1 节的评测协议，固定初始条件区域（在桌面上贴标记）
- [ ] 实验日志、视频录制、失败分类
- [ ] 动作回放测试：把一条示教的动作**原样回放**，确认关节方向、夹爪、单位、时间对齐都正确（7.04 课提到的最常见 bug）
- **过关**：回放示教的成功率接近 100%

### 阶段 1：SFT 基线（1~2 周）

- [ ] 采集 50~200 条高质量示教；**刻意包含"偏了再纠正"的片段**（3.01 课：分布偏移）
- [ ] 计算并检查归一化统计量（q01/q99），检查是否有几乎不动的维度
- [ ] 微调 π0.5，**保存多个 checkpoint，在真机上比较**（loss 最低 ≠ 真机最好，4.03 课）
- [ ] 评测 50 次，做失败分类
- **过关**：成功率与失败分布稳定（两次独立评测结论一致）

### 阶段 2：针对失败分布选方法

- 前段失败为主 → 补数据 / DAgger（人工接管纠正）/ 加腕部相机
- 末段失败为主（对不准、插一半）→ 进入阶段 3

### 阶段 3：RLT 精修（2~4 周）

- [ ] Stage 1：在已有数据上训练 RL token（只需 VLA 自己的数据）
- [ ] 奖励：成功检测（按键标注或视觉分类器）；**先确保奖励信号 100% 可靠**
- [ ] 只在插入阶段启用 actor（接近阶段交给 VLA）
- [ ] 固定小 σ、残差 actor、BC 正则从大到小；人工接管数据进 replay buffer
- [ ] 每 N 次更新做一次固定协议的评测，记录日志
- **过关**：相对 SFT 基线的提升在统计上可信（第 2 节）

### 阶段 4：鲁棒性与长时间运行

- [ ] 换线材、换光照、换初始区域；连续运行 1 小时以上，统计故障（通信、卡顿、RTC 切换）
- [ ] 记录推理延迟分布，确认 RTC 的 $d$ 设置合理（7.04 课）

### 阶段 5（可选）：RECAP / 数据飞轮

- 把阶段 3~4 中自主运行的数据（含失败）+ 人工纠正收集起来，训练价值函数，做优势条件化训练（8.03 课）

---

## 5. 从玩具到真机：本项目各课的对应关系

| 真机问题 | 回到哪一课 |
|---|---|
| 关节方向/夹爪/归一化不对 | 1.01、7.04 |
| 模型在新位置上表现差 | 3.01（分布偏移）、7.03（数据量与预训练） |
| 动作抖动、切换时冲击 | 5.02（时间集成）、6.02（执行步数）、7.04（RTC） |
| 有多种合理做法时犹豫不决 | 3.02、6.02（多模态） |
| loss 降了但真机没变好 | 4.03（过拟合）、7.03（loss 平台期） |
| 最后一厘米插不进去 | 8.02（RLT） |
| 示教质量参差不齐 | 8.03（RECAP） |
| 不确定改进是否真实 | 8.04（置信区间） |

## 小结

- 先定评测协议，再做实验；失败分类比成功率更有指导意义。
- 20 次试验的置信区间很宽，想区分 80% 与 90% 需要 50~100 次。
- 每个阶段设过关标准；用失败分布决定下一步用什么方法。

🎉 **恭喜你完成了 VLA 阶梯的全部内容！** 下一步建议：回到 [第 0 章](../00_guide/0.2_code_thinking.md) 的"三遍法"，挑出你觉得最难的 3 个练习，在空白文件里从零再写一遍。